# Patient Readmission Predictive Modeling
### Clinical Risk Assessment Using Machine Learning
### By Viniciah St. Julien
### Project Overview
Hospital readmissions serve as a critical metric for healthcare quality, patient outcomes, and operational efficiency. Unplanned readmissions not only stress healthcare infrastructure but also signal potential gaps in initial discharge planning or transitional patient care. 

This project utilizes a structured clinical dataset containing demographic and medical information for 5,000 patients sourced from Kaggle. The primary objective is to build and evaluate machine learning models capable of predicting whether a patient will be readmitted (`readmitted = 1`) or not (`readmitted = 0`). By identifying high-risk individuals prior to discharge, healthcare providers can proactively allocate transitional care resources to mitigate readmission rates.

---

### Dataset Profile & Technical Architecture
The dataset consists of 5,000 complete patient records with zero missing values across 8 clinically significant variables, including patient age, hospital metrics (days_in_hospital, num_procedures), health complexity (comorbidity_score), and categorical descriptors (gender, primary_diagnosis, discharge_to).

To ensure rigorous data science practices and prevent data leakage, the notebook is structured across the following phases:
* **Phase 1: Feature Selection Strategy** – Auditing variables for clinical and statistical relevance.
* **Phase 2: Data Splitting** – Isolating an 80% training set and a 20% test set prior to transformations.
* **Phase 3: Categorical Data Inspection** – Auditing unique values within text-based columns to map out encoding requirements.
* **Phase 4: Categorical Encoding** – Converting text-based clinical categories into machine-readable numeric formats via One-Hot Encoding.
* **Phase 5: Model Implementation & Evaluation** – Training and conducting a comparative analysis on three distinct machine learning architectures:
    * *Logistic Regression* (A traditional statistical baseline)
    * *Decision Tree Classifier* (A highly interpretable clinical flowchart model)
    * *Random Forest Classifier* (An advanced ensemble method)

Data Citation:

**Data Citation:**
* **Dataset Title:** Hospital Readmission Dataset
* **Author:** Van Patangan
* **Source:** Sourced via Kaggle API (`vanpatangan/readmission-dataset`)
* **URL:** https://www.kaggle.com/datasets/vanpatangan/readmission-dataset

Import Libraries

To establish a reproducible environment, the required libraries are imported below. The workspace configuration is categorized by functional utility:
* **Data Manipulation:** `pandas` is utilized for dataframe construction, structural indexing, and data handling.
* **Model Deployment Pipeline:** `scikit-learn` dependencies are integrated for data partitioning (`train_test_split`), categorical feature mapping, and executing distinct mathematical frameworks (Logistic Regression, Decision Trees, and Random Forests).
* **Performance Analytics:** Evaluation utilities from `sklearn.metrics` are leveraged to extract granular classification reports to analyze the effects of class imbalance.

In [1]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

Data Loading 

In [2]:
pd.read_csv('train_df.csv')
df = pd.read_csv('train_df.csv')
train = pd.read_csv('train_df.csv')

### Exploratory Data Analysis (EDA)
Before proceeding with feature transformations and model training, a systematic audit of the dataset is conducted to profile data types, structural dimensions, and evaluate the presence of missing values.

In [3]:
train.head()

,age,gender,primary_diagnosis,num_procedures,days_in_hospital,comorbidity_score,discharge_to,readmitted
0,69,Male,Heart Disease,1,2,1,Home Health Care,0
1,32,Female,COPD,2,13,2,Rehabilitation Facility,0
2,89,Male,Diabetes,1,7,1,Home,0
3,78,Male,COPD,9,2,2,Skilled Nursing Facility,0
4,38,Male,Diabetes,6,4,4,Rehabilitation Facility,0


In [4]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   age                5000 non-null   int64 
 1   gender             5000 non-null   object
 2   primary_diagnosis  5000 non-null   object
 3   num_procedures     5000 non-null   int64 
 4   days_in_hospital   5000 non-null   int64 
 5   comorbidity_score  5000 non-null   int64 
 6   discharge_to       5000 non-null   object
 7   readmitted         5000 non-null   int64 
dtypes: int64(5), object(3)
memory usage: 312.6+ KB


In [5]:
# Check for missing values across all features
print("--- Missing Values Per Column ---")
print(df.isnull().sum())

--- Missing Values Per Column ---
age                  0
gender               0
primary_diagnosis    0
num_procedures       0
days_in_hospital     0
comorbidity_score    0
discharge_to         0
readmitted           0
dtype: int64


In [6]:
train.describe()

,age,num_procedures,days_in_hospital,comorbidity_score,readmitted
count,5000.000000,5000.00000,5000.000000,5000.000000,5000.000000
mean,53.299000,4.46100,7.396600,2.068600,0.188000
std,20.646851,2.88606,4.025587,1.422357,0.390751
min,18.000000,0.00000,1.000000,0.000000,0.000000
25%,36.000000,2.00000,4.000000,1.000000,0.000000
50%,53.000000,4.00000,7.000000,2.000000,0.000000
75%,71.000000,7.00000,11.000000,3.000000,0.000000
max,89.000000,9.00000,14.000000,4.000000,1.000000


#### ** Initial Feature Selection Note: An inspection of the 8 variables shows that all columns capture direct clinical or demographic data relevant to patient health and hospital outcomes. No administrative noise (such as IDs or billing codes) is present. Therefore, all 8 features are retained for the initial data split, and statistical feature selection will be performed post-splitting. **

>## Data Splitting:

In [7]:
from sklearn.model_selection import train_test_split

# 1. Separate features (X) and target variable (y)
X = train.drop(columns=['readmitted'])
y = train['readmitted']

# 2. Split into 80% training and 20% testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

# Verify the splits
print(f"Training features shape: {X_train.shape}")
print(f"Testing features shape: {X_test.shape}")

Training features shape: (4000, 7)
Testing features shape: (1000, 7)


#### The data has been split between $4000$ patient records for training, $1000$ for testing, and $7$ starting feature columns

> Categorical Encoding:

In [8]:
# Apply One-Hot Encoding to both the training and testing sets
X_train_encoded = pd.get_dummies(X_train, columns=['gender', 'primary_diagnosis', 'discharge_to'], drop_first=True)
X_test_encoded = pd.get_dummies(X_test, columns=['gender', 'primary_diagnosis', 'discharge_to'], drop_first=True)

# Align the test set with the train set just in case a specific diagnosis 
# only appeared in one of the splits
X_train_encoded, X_test_encoded = X_train_encoded.align(X_test_encoded, join='left', axis=1, fill_value=0)

# Verify that all columns are now numbers
print("New training data columns:")
print(X_train_encoded.columns)
print(f"\nNew shape of training features: {X_train_encoded.shape}")

New training data columns:
Index(['age', 'num_procedures', 'days_in_hospital', 'comorbidity_score',
       'gender_Male', 'primary_diagnosis_Diabetes',
       'primary_diagnosis_Heart Disease', 'primary_diagnosis_Hypertension',
       'primary_diagnosis_Kidney Disease', 'discharge_to_Home Health Care',
       'discharge_to_Rehabilitation Facility',
       'discharge_to_Skilled Nursing Facility'],
      dtype='object')

New shape of training features: (4000, 12)


###  Interpretation of Categorical Encoding 

#### The execution of One-Hot Encoding successfully converted all text-based categorical variables (`gender`, `primary_diagnosis`, and `discharge_to`) into machine-readable numeric formats. As a result of this transformation, the feature space expanded from the original 7 columns to **12 numeric columns**, while maintaining the 4,000 patient records allocated for training.

##### Key Highlights of the Transformation:
* **Prevention of Multi-Collinearity:** By utilizing the `drop_first=True` parameter, the first category of each variable was omitted to act as the baseline reference. For instance, the original `gender` column was compressed into a single binary column (`gender_Male`), where a `1` represents a male patient and a `0` inherently represents a female patient. This effectively prevents the multi-collinearity trap.
* **Feature Expansion Breakdown:** * The `primary_diagnosis` category expanded into 4 distinct binary vectors (Diabetes, Heart Disease, Hypertension, and Kidney Disease), with the remaining omitted category serving as the baseline.
  * The `discharge_to` category expanded into 3 distinct binary vectors representing the care facilities.
* **Data Alignment Verification:** The training and testing matrices were strictly aligned using a left-join configuration. This ensures that even if a rare clinical diagnosis appeared only in the training split, the test split columns remain perfectly synchronized, guaranteeing seamless compatibility for the modeling stage.

This project strictly falls under Supervised Machine Learning: Classification, because we are training models using historical labels to predict a discrete binary outcome (readmitted = 1 or 0).

> ### Model 1: Logistic Regression (The Baseline)

> A baseline model is established using Logistic Regression. In healthcare analytics, Logistic Regression serves as a traditional benchmark because it calculates the probability of clinical risk and offers clear statistical coefficients. This baseline provides a standard score to evaluate whether more complex machine learning algorithms offer an improvement in predictive power.

In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Initialize the Logistic Regression model
# max_iter is set to 1000 to ensure the mathematical solver converges
lr_model = LogisticRegression(max_iter=1000, random_state=42)

# Train the model on the encoded training data
lr_model.fit(X_train_encoded, y_train)

# Generate predictions on the encoded test data
lr_preds = lr_model.predict(X_test_encoded)

# Evaluate the baseline performance
print("=== LOGISTIC REGRESSION PERFORMANCE ===")
print(f"Accuracy Score: {accuracy_score(y_test, lr_preds):.4f}\n")
print(classification_report(y_test, lr_preds))

=== LOGISTIC REGRESSION PERFORMANCE ===
Accuracy Score: 0.8260

              precision    recall  f1-score   support

           0       0.83      1.00      0.90       826
           1       0.00      0.00      0.00       174

    accuracy                           0.83      1000
   macro avg       0.41      0.50      0.45      1000
weighted avg       0.68      0.83      0.75      1000



/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier,


### Interpretation of Baseline Results: The Class Imbalance Trap 


#### The baseline Logistic Regression model achieved an accuracy score of 82.60%. However, a deeper analysis of the classification report reveals a severe class imbalance issue. Out of 1,000 test cases, 826 belong to the negative class (not readmitted) and only 174 belong to the positive class (readmitted).

#### The baseline model fell into a "majority class trap" by predicting 0 (not readmitted) for every single patient. Consequently, while the overall accuracy appears high, the Recall for Class 1 is 0.00%, meaning the model failed to identify a single readmitted patient. In a clinical environment, this model would be unsafe. This highlights why overall accuracy is a misleading metric for imbalanced datasets and justifies the need for non-linear tree-based models.

>### Model 2: Decision Tree Classifier
>Next, a Decision Tree Classifier is implemented. Decision trees are highly valued in clinical workflows because they naturally mirror medical decision-making flowcharts (e.g., separating patients based on a threshold for comorbidity_score or days_in_hospital). Unlike linear models, a decision tree automatically captures non-linear relationships and interactions between clinical variables.

In [10]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report

# Initialize the Decision Tree Classifier
# random_state ensures the splits are reproducible every time the cell runs
dt_model = DecisionTreeClassifier(random_state=42)

# Train the model on the encoded training data
dt_model.fit(X_train_encoded, y_train)

# Generate predictions on the encoded test data
dt_preds = dt_model.predict(X_test_encoded)

# Evaluate the model performance
print("=== DECISION TREE PERFORMANCE ===")
print(f"Accuracy Score: {accuracy_score(y_test, dt_preds):.4f}\n")
print(classification_report(y_test, dt_preds))

=== DECISION TREE PERFORMANCE ===
Accuracy Score: 0.6770

              precision    recall  f1-score   support

           0       0.82      0.78      0.80       826
           1       0.15      0.19      0.17       174

    accuracy                           0.68      1000
   macro avg       0.49      0.48      0.48      1000
weighted avg       0.70      0.68      0.69      1000



#### Interpretation of Decision Tree Results: Trading Accuracy for Clinical Utility ***
The Decision Tree Classifier achieved an overall accuracy score of 67.70%, which is lower than the baseline model. However, unlike the Logistic Regression baseline, the Decision Tree successfully broke out of the majority class trap.

#### The model achieved a Recall of 0.19 for Class 1, meaning it successfully identified 19% of the patients who required readmission. While the model suffers from low precision (0.15)—resulting in a high number of false positives—it represents a significant step forward in clinical utility. The drop in overall accuracy is a classic trade-off when forcing a model to look past a heavy class imbalance.

>Model 3: Random Forest Classifier (Ensemble Method)

>While a single decision tree is highly interpretable, it can be prone to overfitting the training data. To mitigate this risk, a Random Forest Classifier is trained. This ensemble method constructs a multitude of individual decision trees on random subsets of the data and averages their predictions. This approach typically reduces model variance and increases overall generalization accuracy on unseen test data.

In [11]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Initialize the Random Forest Classifier
# n_estimators=100 creates an ensemble of 100 individual decision trees
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)

# Train the model on the encoded training data
rf_model.fit(X_train_encoded, y_train)

# Generate predictions on the encoded test data
rf_preds = rf_model.predict(X_test_encoded)

# Evaluate the ensemble performance
print("=== RANDOM FOREST PERFORMANCE ===")
print(f"Accuracy Score: {accuracy_score(y_test, rf_preds):.4f}\n")
print(classification_report(y_test, rf_preds))

=== RANDOM FOREST PERFORMANCE ===
Accuracy Score: 0.8150

              precision    recall  f1-score   support

           0       0.82      0.99      0.90       826
           1       0.08      0.01      0.01       174

    accuracy                           0.81      1000
   macro avg       0.45      0.50      0.45      1000
weighted avg       0.69      0.81      0.74      1000



#### Interpretation of Random Forest Results: The Ensemble Optimization Trade-off ***

#### The Random Forest Classifier achieved a high overall accuracy score of 81.50%. However, an analysis of the minority class metrics reveals that the ensemble model struggled severely with the dataset's class imbalance. The model achieved a **Recall of only 0.01 for Class 1**, meaning it successfully identified just 1% of the patients who were actually readmitted. 

#### This behavior occurs because random forests optimize for overall accuracy by aggregating the predictions of 100 individual decision trees. When 82.6% of the underlying training data belongs to the majority class (not readmitted), the mathematical penalty for misclassifying the minority class is relatively low. Consequently, the ensemble algorithm defaults to a conservative strategy that leans heavily into predicting the majority class to maximize overall correctness. While this effectively reduces false positives, it makes the model clinically ineffective for proactive patient risk intervention.

In [12]:
# Create a dictionary with the results from all three models
summary_data = {
    'Model': ['Logistic Regression (Baseline)', 'Decision Tree', 'Random Forest (Ensemble)'],
    'Overall Accuracy': [0.8260, 0.6770, 0.8150],
    'Class 1 Precision (Readmitted)': [0.00, 0.15, 0.08],
    'Class 1 Recall (Readmitted)': [0.00, 0.19, 0.01],
    'Class 1 F1-Score': [0.00, 0.17, 0.01]
}

# Convert to DataFrame and display
comparison_df = pd.DataFrame(summary_data)
comparison_df

,Model,Overall Accuracy,Class 1 Precision (Readmitted),Class 1 Recall (Readmitted),Class 1 F1-Score
0,Logistic Regression (Baseline),0.826,0.00,0.00,0.00
1,Decision Tree,0.677,0.15,0.19,0.17
2,Random Forest (Ensemble),0.815,0.08,0.01,0.01


> Final Model Comparison & Clinical Conclusion
> 
>When comparing the three machine learning models, a stark contrast emerges between mathematical accuracy and clinical utility:
> 1. The Accuracy Illusion: Both Logistic Regression (82.60%) and Random Forest (81.50%) yield high overall accuracy scores. However, due to the heavy class imbalance in the dataset, these models achieved high scores simply by overwhelming the positive class and predicting that almost nobody would be readmitted. In a hospital setting, these models are clinically ineffective because their Recall for high-risk patients is essentially $0\%$.
> 2. The Clinical Winner: The Decision Tree Classifier, despite having the lowest overall accuracy (67.70%), is the only model that successfully extracted patterns from the minority class, achieving a Recall of 19% for readmitted patients.